# OpenPlaque — Proximal-Trunk Expanded-Field Reacquisition v1 (execution-fixed)

Starts from the independently dense-QC accepted 20-mm proximal-trunk continuation endpoint and tests whether the previous narrow source-field crop was the remaining technical bottleneck. The frozen master is not modified, and a positive path is not automatically labeled LM.

This notebook revision changes execution/progress handling only. Scientific code remains pinned to the same commit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, time
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR=DRIVE_ROOT+'/Left_Proximal_Trunk_Expanded_Field_Reacquisition_v1'
REUSE_EXISTING_OUTPUTS=False
BRANCH='left-proximal-trunk-expanded-field-reacquisition-from-main'
PINNED_SCIENCE_COMMIT='5ece5c5645463f09b01e561db3375887269d3a5d'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM='left-proximal-trunk-expanded-field-reacquisition-v1.0'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUTPUT_DIR)/'notebook_started.json').write_text(json.dumps({'status':'NOTEBOOK_STARTED','time_unix':time.time(),'branch':BRANCH,'pinned_science_commit':PINNED_SCIENCE_COMMIT},indent=2))
print('Output:', OUTPUT_DIR)
print('Reuse existing outputs:', REUSE_EXISTING_OUTPUTS)
print('Branch:', BRANCH)
print('Pinned science commit:', PINNED_SCIENCE_COMMIT)
print('Heartbeat:', Path(OUTPUT_DIR)/'notebook_started.json')


In [ ]:
import os, shutil, subprocess
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--detach',PINNED_SCIENCE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()
mb=subprocess.check_output(['git','-C',repo,'merge-base','HEAD',BASELINE],text=True).strip()
print('Checked out HEAD:', head)
print('Merge base:', mb)
assert head == PINNED_SCIENCE_COMMIT, (head, PINNED_SCIENCE_COMMIT)
assert mb == BASELINE, (mb, BASELINE)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_proximal_trunk_expanded_field_reacquisition_v1 as exp
print('openplaque:', openplaque.__file__)
print('experiment:', exp.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(), exp.__file__, 'exec')
print('synthetic self-test:', exp.synthetic_expanded_field_self_test())
rc=pytest.main(['-q','/content/OpenPlaque/tests/test_left_proximal_trunk_expanded_field_reacquisition_v1.py'])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
import json
root=Path(DRIVE_ROOT)
required=[
    root/'Left_Proximal_Trunk_Continuation_QC_v1/summary.json',
    root/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
    root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
    root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
    root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
    root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:', len(required), 'missing:', len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
prior=json.loads(required[0].read_text())
print('Prior status:', prior.get('status'))
print('Prior accepted continuation mm:', prior.get('candidate_dense_qc',{}).get('accepted_arc_mm'))
assert prior.get('status') == 'PROXIMAL_TRUNK_CONTINUATION_QC_POSITIVE'
(Path(OUTPUT_DIR)/'preflight_complete.json').write_text(json.dumps({'status':'PREFLIGHT_COMPLETE','prior_status':prior.get('status')},indent=2))


In [ ]:
import time, gc
from openplaque.left_proximal_trunk_expanded_field_reacquisition_v1 import run
gc.collect(); t0=time.time()
result=run(DRIVE_ROOT, OUTPUT_DIR)
s=result['summary']
print('ELAPSED MIN:', round((time.time()-t0)/60,2))
print('STATUS:', s.get('status'))
print('RCA CONTROL PASS:', s.get('RCA_plane_qc_control',{}).get('accepted'))
print('START AORTA DIST MM:', s.get('search_budget',{}).get('start_aorta_distance_mm'))
print('REACHED AORTA:', s.get('expanded_field',{}).get('reached_aorta'))
print('BEST AORTA DIST MM:', s.get('expanded_field',{}).get('best_aorta_distance_mm'))
print('DENSE-QC ACCEPTED EXTENSION MM:', s.get('dense_qc',{}).get('accepted_arc_mm'))
print('RCA-INDEPENDENT ENDPOINT:', s.get('posthoc_RCA_independent'))
print('REPORT:', result.get('report'))
print('ZIP:', result.get('zip'))
